# 🎤 AI-RVC 语音转换 & AI 翻唱 (Colab)

基于 RVC v2 + RMVPE 的高质量语音转换系统。

**使用前请确保：**
- 运行时类型已设置为 **GPU**（菜单栏 → 代码执行程序 → 更改运行时类型 → T4 GPU）
- 按顺序执行下方每个单元格

---

## 功能特点

- **AI 歌曲翻唱**：上传歌曲自动分离人声、转换音色、混合伴奏
- **人声分离**：Mel-Band Roformer (KimberleyJensen) - MVSEP Vocals SDR 11.01 / Instrum SDR 17.32
- **语音转换**：RVC v2 + FAISS 检索增强
- **RMVPE 音高提取**：高精度 F0 提取，噪声鲁棒性强
- **角色模型**：内置 117 个可下载角色模型
- **混音效果**：支持人声混响、音量调节、混音预设
- **卡拉OK模式**：分离主唱和伴唱轨道

---

## 1️⃣ 环境检查

检查 GPU 和 Python 环境

In [ ]:
# 检查 GPU
!nvidia-smi

# 检查 Python 版本
import sys
print(f"Python 版本: {sys.version}")

# 检查 CUDA
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2️⃣ 克隆仓库

从 GitHub 克隆 AI-RVC 项目

In [ ]:
# 克隆仓库
!git clone https://github.com/mason369/AI-RVC.git
%cd AI-RVC

# 显示项目结构
!ls -la

## 3️⃣ 安装依赖

安装 PyTorch 和项目依赖

In [ ]:
# 安装 PyTorch (CUDA 12.1)
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

# 安装项目依赖
!pip install -r requirements.txt

# 验证安装
import torch
print(f"\n✅ PyTorch 安装成功: {torch.__version__}")
print(f"✅ CUDA 可用: {torch.cuda.is_available()}")

## 4️⃣ 下载基础模型

下载 HuBERT、RMVPE 等必需模型

In [ ]:
# 下载基础模型
!python tools/download_models.py

# 检查模型状态
from tools.download_models import check_model, REQUIRED_MODELS

print("\n基础模型状态:")
for name in REQUIRED_MODELS:
    status = "✅" if check_model(name) else "❌"
    print(f"{status} {name}")

## 5️⃣ 下载角色模型（可选）

下载预训练的角色模型。你可以：
- 下载单个角色
- 下载指定系列的所有角色
- 下载全部 117 个角色（需要较长时间）

In [ ]:
# 查看可用角色列表
from tools.character_models import list_available_characters, list_available_series

print("可用系列:")
for series in list_available_series():
    print(f"  - {series}")

print("\n可用角色（前 20 个）:")
chars = list_available_characters()
for char in chars[:20]:
    print(f"  - {char['name']}: {char.get('display', char.get('description', ''))} (出自: {char.get('source', '未知')})")

print(f"\n总计: {len(chars)} 个角色")

In [ ]:
# 下载单个角色（示例：星空凛）
from tools.character_models import download_character_model

# 修改这里的角色名称来下载不同的角色
character_name = "rin"  # 星空凛

print(f"正在下载角色: {character_name}")
success = download_character_model(character_name)
if success:
    print(f"✅ {character_name} 下载完成")
else:
    print(f"❌ {character_name} 下载失败")

In [ ]:
# 下载指定系列的所有角色（示例：Love Live!）
from tools.character_models import download_all_character_models

# 修改这里的系列名称来下载不同系列
series_name = "Love Live!"  # 可选: "Love Live! Sunshine!!", "原神", "Hololive" 等

print(f"正在下载系列: {series_name}")
result = download_all_character_models(series=series_name)
print(f"✅ 成功: {len(result['success'])} 个")
if result['failed']:
    print(f"❌ 失败: {len(result['failed'])} 个: {', '.join(result['failed'])}")

In [ ]:
# 下载全部角色（需要较长时间，约 10-20 分钟）
# 取消注释下面的代码来执行

# from tools.character_models import download_all_character_models
# print("正在下载全部 117 个角色模型...")
# result = download_all_character_models()
# print(f"✅ 成功: {len(result['success'])} 个")
# if result['failed']:
#     print(f"❌ 失败: {len(result['failed'])} 个: {', '.join(result['failed'])}")

## 6️⃣ 启动 Gradio 界面

启动 Web 界面，支持所有功能

In [ ]:
# 启动 Gradio 界面
!python run.py --host 0.0.0.0 --port 7860 --share --skip-check

## 7️⃣ 命令行翻唱（可选）

如果你想通过代码直接处理翻唱，可以使用下面的方法

In [ ]:
# 导入必要的模块
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from infer.cover_pipeline import get_cover_pipeline
from tools.character_models import get_character_model_path

# 配置参数
input_audio = "/content/your_song.mp3"  # 修改为你的歌曲路径
character_name = "rin"  # 修改为你下载的角色名称
output_dir = "/content/outputs"

# 获取角色模型路径
model_info = get_character_model_path(character_name)
if model_info is None:
    print(f"❌ 角色模型不存在: {character_name}")
    print("请先下载角色模型（参见上面的单元格）")
else:
    print(f"✅ 找到角色模型: {character_name}")
    print(f"模型路径: {model_info['model_path']}")
    
    # 获取翻唱流水线
    pipeline = get_cover_pipeline("cuda")
    
    # 执行翻唱
    print("\n开始处理翻唱...")
    result = pipeline.process(
        input_audio=input_audio,
        model_path=model_info["model_path"],
        index_path=model_info.get("index_path"),
        pitch_shift=0,  # 音高偏移（-12 ~ 12）
        index_ratio=0.35,  # 索引率（0 ~ 1）
        filter_radius=3,
        rms_mix_rate=0.15,
        protect=0.33,
        speaker_id=0,
        f0_method="rmvpe",
        separator="roformer",  # 人声分离器：roformer/demucs/uvr5
        vocals_volume=1.0,  # 人声音量（0 ~ 2）
        accompaniment_volume=1.0,  # 伴奏音量（0 ~ 2）
        reverb_amount=0.1,  # 混响量（0 ~ 1）
        karaoke_separation=True,  # 启用卡拉OK分离
        karaoke_merge_backing_into_accompaniment=True,  # 合并伴唱到伴奏
        vc_preprocess_mode="auto",  # VC预处理模式：auto/direct/uvr_deecho/legacy
        source_constraint_mode="auto",  # 源约束模式：auto/off/on
        vc_pipeline_mode="current",  # VC管道模式：current/official
        singing_repair=False,  # 唱歌修复（仅official模式）
        output_dir=output_dir,
        model_display_name=character_name
    )
    
    print("\n✅ 翻唱完成！")
    print(f"最终翻唱: {result['cover']}")
    print(f"转换后人声: {result['converted_vocals']}")
    print(f"原始人声: {result.get('vocals', 'N/A')}")
    print(f"主唱: {result.get('lead_vocals', 'N/A')}")
    print(f"伴唱: {result.get('backing_vocals', 'N/A')}")
    print(f"伴奏: {result['accompaniment']}")
    print(f"\n所有文件目录: {result.get('all_files_dir', 'N/A')}")

## 8️⃣ 下载输出文件

将生成的翻唱文件下载到本地

In [ ]:
# 列出输出目录中的文件
!ls -lh outputs/

# 使用 Colab 的文件下载功能
from google.colab import files

# 下载最终翻唱（修改文件名）
# files.download('outputs/your_cover.wav')

---

## 📝 参数说明

### 转换参数

| 参数 | 说明 | 建议值 |
|------|------|--------|
| `pitch_shift` | 音调偏移（半音数） | 男转女: +12, 女转男: -12 |
| `index_ratio` | 索引比率（越高越像训练音色） | 0.1-0.5 |
| `filter_radius` | 中值滤波（减少气音抖动） | 3 |
| `protect` | 保护系数（防止撕裂伪影） | 0.33 |
| `rms_mix_rate` | RMS 混合率（音量包络匹配） | 0.15 |

### 混音参数

| 参数 | 说明 | 建议值 |
|------|------|--------|
| `vocals_volume` | 人声音量 | 1.0 (100%) |
| `accompaniment_volume` | 伴奏音量 | 1.0 (100%) |
| `reverb_amount` | 人声混响 | 0.1-0.2 |
| `backing_mix` | 伴唱混合率 | 0.0-1.0 |

### VC 预处理模式

| 模式 | 说明 |
|------|------|
| `auto` | 自动选择（推荐） |
| `direct` | 主唱直通 RVC |
| `uvr_deecho` | 使用学习型 DeEcho/DeReverb |
| `legacy` | 旧版手工链（仅用于对比） |

### 人声分离器

| 分离器 | 说明 |
|--------|------|
| `roformer` | Mel-Band Roformer（默认，质量最高） |
| `demucs` | Demucs（速度较快） |
| `uvr5` | UVR5（兼容性好） |

---

## 🔧 常见问题

**Q: CUDA out of memory**

A: 尝试使用较短的音频（< 5 分钟），或切换到 `demucs` 分离器

**Q: 高音断音/撕裂**

A: 降低 `protect` 系数（0.33 → 0.2），增大 `filter_radius`（3 → 5）

**Q: 转换后声音失真**

A: 降低 `index_ratio`，调整 `pitch_shift`，使用更高质量的输入音频

---

## 📚 更多信息

- GitHub: https://github.com/mason369/AI-RVC
- 完整文档: 查看仓库中的 README.md
- 角色模型: 117 个可下载角色（Love Live!, 原神, Hololive 等）

---

## ⚠️ 免责声明

本项目仅供学习研究和个人娱乐用途，不得用于任何商业目的。严禁使用本软件进行欺诈、传播虚假信息或侵犯他人权益。用户对使用本软件产生的所有内容和后果承担全部责任。